# RAG Bench — 72-Combo Benchmark on Google Colab

한국어 RAG 파이프라인 72개 조합을 Google Colab T4 GPU에서 벤치마크합니다.

## 3-Layer Architecture
```
Layer 1: Dense Model ─── kosimcse | e5 | bge-m3 | minilm       (4종)
Layer 2: Sparse Model ── korean_bm25 | splade | fastembed_bm25 (3종)
Layer 3: Retrieval Mode ─ hybrid × reranker × llm_support      (6종)
                          ├── hybrid (기본)
                          ├── hybrid + contextual
                          ├── hybrid + colbert_rerank
                          ├── hybrid + colbert_rerank + contextual
                          ├── hybrid + flashrank_rerank
                          └── hybrid + flashrank_rerank + contextual

총 조합: 4 × 3 × 6 = 72개 + GraphRAG
```

## 2-Pass Execution
- **Pass 1**: 전체 조합 레이턴시 측정 (API 비용 $0)
- **Pass 2**: 상위 N개만 RAGAS 평가 (API 비용 절감)
  - **메트릭 프리셋**: `core_only` (4) | `comprehensive` (7) | `full` (11+) | `reference_free`
  - **스코어링 프로파일**: `balanced` | `precision_critical` | `speed_critical` | `comprehensive`

## 예상 실행시간 (T4 GPU)
| 프리셋 | 조합 수 | Pass 1 | Pass 2 | 총 예상 |
|--------|---------|--------|--------|--------|
| quick | 4 | ~3분 | ~10분 | ~15분 |
| standard | 24 | ~20분 | ~30분 | ~50분 |
| full | 72 | ~1시간 | ~2시간 | ~3시간 |
| +GraphRAG | +1 | +5분 | +10분 | +15분 |

---
## Section 1: 환경 설정

In [ ]:
# Cell 1.1: Repo clone + sys.path 설정
import os
import sys

REPO_URL = "https://github.com/SukbeomH/RAG-Bench.git"  # <- 본인 repo URL로 변경
REPO_DIR = "/content/RAG-Bench"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned: {REPO_DIR}")

# sys.path에 추가 (패키지 import용)
for p in [REPO_DIR, os.path.join(REPO_DIR, "rag_bench_colab")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"sys.path 설정 완료")
print(f"Python: {sys.version}")

In [ ]:
# Cell 1.2: 시스템 의존성 + pip install
!apt-get install -y default-jdk -qq
!pip install -q -r /content/RAG-Bench/rag_bench_colab/requirements_colab.txt

In [ ]:
# Cell 1.3: Colab 환경 초기화 + rag_bench 패치 + smoke test
from colab_config import init_colab

env_info = init_colab(
    qdrant_mode="ephemeral",  # 'ephemeral' | 'drive' | 'memory'
    device=None,               # None = 자동 감지 (T4 → 'cuda')
    mount_drive=True,
)

# Smoke test: 핵심 모듈 import
from rag_bench.config import BENCH_DATA_DIR, BENCH_DOCS_DIR
from rag_bench.scripts.run_all_combos import PRESETS, generate_valid_combinations, ComboSpec
from rag_bench.runner import BenchmarkRunner
print("\n[Smoke Test] 모든 import 성공!")
print(f"  BENCH_DATA_DIR: {BENCH_DATA_DIR}")
print(f"  BENCH_DOCS_DIR: {BENCH_DOCS_DIR}")

---
## Section 2: 설정

### QDRANT_MODE 선택 가이드

| 모드 | 저장 위치 | 세션 종료 후 | 추천 상황 |
|------|-----------|-------------|-----------|
| `ephemeral` | `/content/qdrant_workspace` (로컬) | **삭제됨** | 빠른 테스트, 재현 불필요 |
| `drive` | `Google Drive/MyDrive/rag_bench_colab/` | **유지됨** | 장시간 실험, 결과 보존 필요 |
| `memory` | 메모리 (RAM) | **삭제됨** | 가장 빠름, 소규모 실험 |

> **권장**: 처음 실행은 `ephemeral`, 결과를 보존하려면 `drive`
>
> `drive` 모드는 Cell 1.3의 `mount_drive=True` 필요 (Google Drive 마운트)

### 기타 파라미터

| 파라미터 | 설명 |
|----------|------|
| `PRESET` | 벤치마크 조합 수 — `quick` (4) / `standard` (24) / `full` (72) |
| `K` | 검색 시 반환할 문서 수 |
| `TOP_N` | Pass 1 완료 후 RAGAS 평가할 상위 전략 수 |
| `METRIC_PRESET` | 평가 메트릭 세트 — `core_only` / `comprehensive` / `full` / `reference_free` |
| `SCORING_PROFILE` | 가중 점수 프로파일 — `balanced` / `precision_critical` / `speed_critical` / `comprehensive` |

In [ ]:
# ===== 사용자 설정 =====
PRESET = "quick"     # 'quick' (4조합) | 'standard' (24) | 'full' (72)
K = 3                # 검색 결과 수
TOP_N = 4            # Pass 2 RAGAS 평가 대상 (상위 N)

# QDRANT_MODE:
#   'ephemeral' → /content/qdrant_workspace  (세션 종료 시 삭제, 기본값)
#   'drive'     → Google Drive/rag_bench_colab/ (영구 보존, mount_drive=True 필요)
#   'memory'    → RAM 전용 (가장 빠름, 세션 종료 시 삭제)
QDRANT_MODE = "ephemeral"

RUN_GRAPHRAG = False  # GraphRAG 포함 여부 (LLM API 비용 발생)

# 평가 설정 (rag_bench evaluation 최신화 반영)
METRIC_PRESET = "core_only"    # 'core_only' (4) | 'comprehensive' (7) | 'full' (11+) | 'reference_free'
SCORING_PROFILE = "balanced"   # 'balanced' | 'precision_critical' | 'speed_critical' | 'comprehensive'
# ======================

print(f"Preset: {PRESET}")
print(f"K: {K}, Top-N: {TOP_N}")
print(f"Qdrant Mode: {QDRANT_MODE}")
print(f"GraphRAG: {'ON' if RUN_GRAPHRAG else 'OFF'}")
print(f"Metric Preset: {METRIC_PRESET}")
print(f"Scoring Profile: {SCORING_PROFILE}")

---
## Section 3: 데이터 로딩

In [ ]:
# Cell 3.1: 러너 생성 + 데이터 로드
from colab_runner import ColabBenchmarkRunner

runner = ColabBenchmarkRunner(
    preset=PRESET,
    k=K,
    top_n=TOP_N,
    qdrant_mode=QDRANT_MODE,
    metric_preset=METRIC_PRESET,
    scoring_profile=SCORING_PROFILE,
)

child_chunks, parent_pairs, queries, ground_truths = runner.prepare_data()

print(f"\nQA 샘플:")
for i, q in enumerate(queries[:3]):
    print(f"  Q{i+1}: {q[:80]}...")
    print(f"  A{i+1}: {ground_truths[i][:80]}...")

In [ ]:
# Cell 3.2: Parent-Child 청킹 통계
print(f"Parent 청크: {len(parent_pairs)}개")
print(f"Child 청크: {len(child_chunks)}개")
print(f"\n샘플 Child 청크 (첫 번째):")
print(child_chunks[0].page_content[:300])

---
## Section 4: 조합 생성

In [ ]:
# 프리셋 기반 ComboSpec 생성
combos = runner.generate_combos()

import pandas as pd
combo_table = pd.DataFrame([
    {
        "#": i+1,
        "Label": spec.label,
        "Dense": spec.dense,
        "Sparse": spec.sparse,
        "Reranker": spec.reranker or "-",
        "LLM Support": spec.llm_support or "-",
    }
    for i, spec in enumerate(combos)
])
display(combo_table)

---
## Section 5: Pass 1 — 레이턴시 벤치마크

In [ ]:
# Pass 1: 전체 조합 레이턴시 측정
latency_df = runner.run_pass1(combos, queries, child_chunks, parent_pairs)
display(latency_df)

In [ ]:
# Pass 1 시각화
from colab_visualizer import plot_latency_comparison
plot_latency_comparison(latency_df)

---
## Section 6: Pass 2 — RAGAS 평가

In [ ]:
# Pass 2: 상위 N개 전략 RAGAS 평가 (체크포인트 지원)
ragas_df = runner.run_pass2(
    latency_df, combos, queries, ground_truths,
    child_chunks, parent_pairs,
)
display(ragas_df)

In [ ]:
# RAGAS 결과 스타일링 테이블
from colab_visualizer import display_styled_table, display_weighted_scores
display_styled_table(ragas_df)

# Weighted Score (프로파일별 가중 점수)
if runner.reports:
    print(f"\n--- Weighted Scores (profile={SCORING_PROFILE}) ---")
    display_weighted_scores(runner.reports, scoring_profile=SCORING_PROFILE)

---
## Section 7: GraphRAG 벤치마크 (선택사항)

In [ ]:
# GraphRAG 실행 (RUN_GRAPHRAG=True 일 때만)
graphrag_result = None
if RUN_GRAPHRAG:
    graphrag_result = runner.run_graphrag(parent_pairs, queries, ground_truths)
    if graphrag_result and "ragas" in graphrag_result:
        print("\nGraphRAG RAGAS 결과:")
        for k, v in graphrag_result["ragas"].items():
            print(f"  {k}: {v}")
else:
    print("GraphRAG 건너뜀 (RUN_GRAPHRAG=False)")

In [ ]:
# GraphRAG 결과를 기존 DataFrame에 추가
if graphrag_result and "ragas" in graphrag_result:
    import pandas as pd
    grag_row = pd.DataFrame([graphrag_result["ragas"]])
    ragas_df = pd.concat([ragas_df, grag_row], ignore_index=True)
    print("GraphRAG 결과 추가 완료")
    display(ragas_df)

---
## Section 8: 시각화 대시보드

수행 이력(RunTracker) + RAGAS 차트 + 가중 점수(Weighted Score) 통합 시각화

In [ ]:
# 수행 이력 요약 (RunTracker 연동)
run_record = runner.get_run_record()
if run_record:
    from colab_visualizer import plot_run_info, plot_phase_timeline
    print("--- Run Summary ---")
    plot_run_info(run_record)
    print("\n--- Phase Timeline ---")
    plot_phase_timeline(run_record)
else:
    print("RunTracker 데이터가 없습니다.")

In [ ]:
# 레이더 차트 (RAGAS 메트릭)
from colab_visualizer import plot_ragas_radar
plot_ragas_radar(ragas_df, top_n=min(5, len(ragas_df)))

In [ ]:
# 히트맵 (전략 x 메트릭)
from colab_visualizer import plot_ragas_heatmap
plot_ragas_heatmap(ragas_df)

In [ ]:
# 파레토 프론티어 (레이턴시 vs 품질)
from colab_visualizer import plot_latency_vs_quality
plot_latency_vs_quality(latency_df, ragas_df)

In [ ]:
# 레이어별 기여도 분석
from colab_visualizer import plot_layer_contribution
plot_layer_contribution(combos, latency_df, metric="avg_latency")

---
## Section 9: 비용 요약 + 결과 저장

In [ ]:
# 비용 요약 (추정)
n_ragas_strategies = len(ragas_df) if ragas_df is not None else 0
n_queries = len(queries)
est_ragas_cost = n_ragas_strategies * n_queries * 0.005  # ~$0.005/query/strategy
est_graphrag_cost = 0.50 if graphrag_result else 0.0

cost_data = {
    "RAGAS 평가 (GPT-4o-mini)": est_ragas_cost,
    "Answer 생성 (GPT-3.5-turbo)": n_ragas_strategies * n_queries * 0.001,
    "GraphRAG (LLM)": est_graphrag_cost,
}
total_cost = sum(cost_data.values())

print(f"예상 API 비용:")
for k, v in cost_data.items():
    print(f"  {k}: ${v:.2f}")
print(f"  총계: ${total_cost:.2f}")

from colab_visualizer import plot_cost_breakdown
if total_cost > 0:
    plot_cost_breakdown(cost_data)

In [ ]:
# 결과 Export (Google Drive)
output_dir = runner.export_results(
    latency_df=latency_df,
    ragas_df=ragas_df,
    graphrag_result=graphrag_result,
)
print(f"\n결과 저장 위치: {output_dir}")